# 🚀 Fabrika GPU İşçisi — Kaggle / Colab (Gerçek FLUX.2 + Wan2.1)

Bu notebook, `yusasufhin.com` video fabrikasına **gerçek difüzyon** yapan bir GPU
işçisi bağlar. Sahte ffmpeg kutusu üretmez; `gpu_worker.py` çalıştırır.

## Adımlar
1. **Kaggle:** `Settings → Accelerator → GPU T4 x2` · **Colab:** `Runtime → Change runtime type → T4 GPU`.
2. İşçi anahtarını `FABRIKA_WORKER_KEY` olarak tanımlayın:
   - Kaggle: `Add-ons → Secrets` içine `FABRIKA_WORKER_KEY` ekleyin.
   - Colab: 🔑 (Secrets) panelinden `FABRIKA_WORKER_KEY` ekleyin.
   - Ya da hücre çalışırken `getpass` soracaktır.
3. Hücreleri sırayla çalıştırın. İşçi sunucudan iş alır, üretir ve geri yükler.

> **VRAM dürüstlüğü:** T4 16 GB'da FLUX.2-klein-4B ve Wan2.1-T2V-1.3B sığar.
> Wan2.1-I2V-14B, `bitsandbytes` NF4 4-bit kuantizasyonla offload'suz sığar;
> kuantizasyon yoksa `enable_sequential_cpu_offload()` ile yavaş yola düşer.
> Sığmazsa işçi dürüst hata bildirir, sahte çıktı üretmez.


In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO = "https://github.com/sufhinyusa/yusasufhin-cockpit.git"
if not Path("gpu_worker.py").is_file():
    if not Path("yusasufhin-cockpit").is_dir():
        subprocess.run(["git", "clone", "--depth", "1", REPO], check=True)
    os.chdir("yusasufhin-cockpit")

# GPU bağımlılıkları (torch/diffusers/transformers/requests)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-gpu.txt"], check=True)
print("Çalışma dizini:", Path.cwd())
print("gpu_worker.py:", Path("gpu_worker.py").is_file())


In [ ]:
import os
from getpass import getpass

WORKER_KEY = (
    os.environ.get("FABRIKA_WORKER_KEY")
    or os.environ.get("WORKER_KEY")
    or os.environ.get("FABRIKA_TOKEN")
)

if not WORKER_KEY:
    try:  # Kaggle Secrets
        from kaggle_secrets import UserSecretsClient
        WORKER_KEY = UserSecretsClient().get_secret("FABRIKA_WORKER_KEY")
    except Exception:
        pass

if not WORKER_KEY:
    try:  # Colab Secrets
        from google.colab import userdata
        WORKER_KEY = userdata.get("FABRIKA_WORKER_KEY")
    except Exception:
        pass

if not WORKER_KEY:
    WORKER_KEY = getpass("FABRIKA_WORKER_KEY: ").strip()

if not WORKER_KEY:
    raise SystemExit("FABRIKA_WORKER_KEY yok — işçi kimlik doğrulaması yapılamaz.")

os.environ["FABRIKA_WORKER_KEY"] = WORKER_KEY
os.environ.setdefault("FABRIKA_SUNUCU", "https://yusasufhin.com")
print("Anahtar: ayarlı (Bearer + X-Worker-Key)")
print("Sunucu:", os.environ["FABRIKA_SUNUCU"])


In [ ]:
# GPU + model kullanılabilirliğini raporla (iş almadan çıkar)
import os, subprocess, sys

subprocess.run([
    sys.executable, "gpu_worker.py",
    "--server", os.environ["FABRIKA_SUNUCU"],
    "--token", os.environ["FABRIKA_WORKER_KEY"],
    "--gpu-kontrol",
], check=False)


In [ ]:
# Gerçek işçi döngüsü — Ctrl/Cmd+C ile durdurun
import os, subprocess, sys

subprocess.run([
    sys.executable, "gpu_worker.py",
    "--server", os.environ["FABRIKA_SUNUCU"],
    "--token", os.environ["FABRIKA_WORKER_KEY"],
    "--kalite", "480p",
], check=False)
